# 02 — Preprocessing
## Sentinels, Missing Data, and Split-Safe Cleaning

**Credit Scoring Capstone — Module 1**

The EDA found three ways this data misleads: sentinel values masquerading as measurements, a 47-column building-info block that is half empty, and missingness that itself predicts default. This notebook turns those findings into a disciplined pipeline:

1. **Fix what lies** — sentinels → NaN + indicator flags (`src/cleaning.py`, idempotent).
2. **Split before anything is fitted** — stratified 70/15/15; the test set is sealed until notebook 05.
3. **Decide the imputation strategy per missingness type** — and prove why imputers must be fit on the training split only.
4. **Persist clean splits with NaNs intact** — the final imputation lives *inside* the model pipelines (notebook 04), so it is fitted on train folds only, serialized with the model, and impossible to leak. This is the production pattern OSFI-style validators expect: one artifact that carries every transform.

In [1]:
import sys, warnings, json
sys.path.append("..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config, data, cleaning, plotting
from src.plotting import (BLUE, ORANGE, AQUA, MUTED, INK_2, GRID, GOOD, CRITICAL,
                          fmt_pct, subtitle, save_figure)

plotting.apply_style()
config.set_seed()

app = data.load_application()
app = cleaning.fix_sentinels(app)
app = cleaning.add_readable_units(app)
print(f"cleaned application table: {app.shape[0]:,} x {app.shape[1]}")

cleaned application table: 307,511 x 126


---
## 1. The Sentinel Audit — Every Column, Not Just the Famous One

`DAYS_EMPLOYED` was caught in the EDA. A systematic audit checks every numeric column for suspicious value spikes (a single value covering > 5% of rows at a distribution extreme).

In [2]:
suspects = []
num = app.select_dtypes(include=[np.number])
for c in num.columns:
    if c in ("SK_ID_CURR", "TARGET") or c.startswith(("FLAG_", "REG_", "LIVE_")):
        continue
    vc = num[c].value_counts(dropna=True)
    if len(vc) < 3:
        continue
    top_val, top_n = vc.index[0], vc.iloc[0]
    share = top_n / num[c].notna().sum()
    # a spike at an extreme (min or max) covering >5% of rows, excluding a natural zero
    if share > 0.05 and top_val != 0 and (top_val == num[c].max() or top_val == num[c].min()):
        suspects.append({"column": c, "value": top_val, "share": round(float(share), 3)})

if suspects:
    display(pd.DataFrame(suspects).sort_values("share", ascending=False).head(10))
else:
    print("No further sentinel-like spikes found - the DAYS_EMPLOYED fix was the only one needed.")

No further sentinel-like spikes found - the DAYS_EMPLOYED fix was the only one needed.


**Audit verdict:** after the `DAYS_EMPLOYED` fix, the audit comes back clean — no other numeric column hides a sentinel spike at a distribution extreme. The cell stays in the pipeline as a reusable guard any new data drop gets run through before modeling.

---
## 2. Split First — Then Fit Anything

70% train / 15% validation / 15% test, stratified on `TARGET` so each split carries the same 8.07% default rate. Everything fitted from here on (imputers, encoders, scalers, cluster centroids, models) sees **training rows only**.

In [3]:
train, val, test = data.stratified_split(app)
summary = pd.DataFrame({
    "rows": [len(train), len(val), len(test)],
    "default_rate": [d["TARGET"].mean() for d in (train, val, test)],
    "defaults": [int(d["TARGET"].sum()) for d in (train, val, test)],
}, index=["train", "validation", "test"])
summary.round(4)

,rows,default_rate,defaults
train,215257,0.0807,17377
validation,46127,0.0807,3724
test,46127,0.0807,3724


---
## 3. Missingness Strategy — by Cause, Not One-Size-Fits-All

The EDA showed three different *kinds* of absence. Each gets its own treatment:

| Missingness type | Examples | Treatment | Why |
|---|---|---|---|
| **Structural block** | 47 building-info columns (~50% missing) | median impute + one shared `BUILDING_INFO_MISSING` flag | The block is absent together; 47 separate flags would be noise |
| **Informative absence** | `EXT_SOURCE_1/3`, `OCCUPATION_TYPE`, `OWN_CAR_AGE` | impute + per-column indicator | The EDA proved absence predicts default |
| **Sentinel-derived** | `DAYS_EMPLOYED` (pensioners) | already flagged by `FLAG_NOT_EMPLOYED` | The flag carries the real signal |
| **Categorical gaps** | `NAME_TYPE_SUITE`, `OCCUPATION_TYPE` | explicit `"Missing"` category | Trees/encoders handle it as its own level — no fake mode |

The *fitting* of imputers happens inside the notebook-04 model pipelines. Here we prove the leakage point that makes that discipline matter:

In [4]:
# Why fit-on-train matters: imputation values differ across splits, and using
# full-data statistics would leak validation/test information into training.
demo_cols = ["EXT_SOURCE_1", "OWN_CAR_AGE", "AMT_ANNUITY", "EMPLOYED_YEARS"]
demo = pd.DataFrame({
    "train median": train[demo_cols].median(),
    "val median": val[demo_cols].median(),
    "full-data median (leaky)": app[demo_cols].median(),
    "missing share (train)": train[demo_cols].isna().mean(),
}).round(3)
demo

,train median,val median,full-data median (leaky),missing share (train)
EXT_SOURCE_1,0.505,0.506,0.506,0.563
OWN_CAR_AGE,9.000,9.000,9.000,0.660
AMT_ANNUITY,24903.000,24880.500,24903.000,0.000
EMPLOYED_YEARS,4.509,4.507,4.512,0.179


In [5]:
# The one transform applied HERE (it is stateless, no fitting): the shared
# building-block missingness flag, computed row-wise from the block itself.
building_cols = [c for c in app.columns if c.endswith(("_AVG", "_MODE", "_MEDI"))]

def add_block_flag(df):
    out = df.copy()
    out["BUILDING_INFO_MISSING"] = out[building_cols].isna().all(axis=1).astype("int8")
    return out

train, val, test = map(add_block_flag, (train, val, test))
print(f"building block columns: {len(building_cols)}")
print(f"share with the whole block missing (train): {train['BUILDING_INFO_MISSING'].mean():.1%}")

rate_by_flag = train.groupby("BUILDING_INFO_MISSING")["TARGET"].mean()
print(f"default rate | block present: {rate_by_flag[0]:.2%}   | block missing: {rate_by_flag[1]:.2%}")

building block columns: 47
share with the whole block missing (train): 47.4%
default rate | block present: 7.06%   | block missing: 9.20%


---
## 4. Persist the Clean Splits

NaNs are kept (pipelines impute them later); sentinels are fixed; readable units and flags are added. A JSON manifest records row counts and the exact column list so notebook 03 can verify it received what notebook 02 produced.

In [6]:
out = config.PROCESSED_DIR
out.mkdir(parents=True, exist_ok=True)
train.to_parquet(out / "train_clean.parquet", index=False)
val.to_parquet(out / "val_clean.parquet", index=False)
test.to_parquet(out / "test_clean.parquet", index=False)

manifest = {
    "rows": {"train": len(train), "val": len(val), "test": len(test)},
    "default_rate": {k: float(d["TARGET"].mean()) for k, d in
                     [("train", train), ("val", val), ("test", test)]},
    "n_columns": train.shape[1],
    "columns": train.columns.tolist(),
}
with open(out / "manifest_02.json", "w") as f:
    json.dump(manifest, f, indent=2)

for p in ["train_clean.parquet", "val_clean.parquet", "test_clean.parquet"]:
    print(f"  {p:24s} {(out / p).stat().st_size / 1e6:7.1f} MB")

  train_clean.parquet         16.7 MB
  val_clean.parquet            4.1 MB
  test_clean.parquet           4.1 MB


---
## 5. What Notebook 03 Receives

- Three stratified splits, sentinel-free, with `FLAG_NOT_EMPLOYED`, `BUILDING_INFO_MISSING`, and readable year units added; NaNs deliberately intact.
- A missingness strategy table (above) that notebook 04's pipelines implement verbatim.
- The leakage argument, demonstrated with numbers: full-data imputation statistics differ from train-only statistics — the gap is small here, but the *discipline* is what a validator audits.

**Next:** notebook `03_feature_engineering.ipynb` — capacity ratios, 1.7M bureau rows collapsed into Character features, categorical encoding, and K-Means peer groups.